# Multi-Output Prediction of Cognition from Brain Imaging

This project trains a deep neural network that takes a subject's 3D brain
imaging data and predicts **several cognitive and behavioral outcomes at
once** (e.g. crystallized cognition score and working-memory task accuracy)
from a single forward pass, using one shared brain representation.

Predicting multiple outcomes jointly, instead of training a separate model
per outcome, is the central idea demonstrated here: the network learns one
shared "bottleneck" representation of the brain scan, then branches into a
small dedicated head per outcome. This is more data-efficient than N
separate single-output models and lets us ask which brain-derived features
are shared across outcomes versus outcome-specific.

**What this notebook covers:**
1. Problem and dataset
2. Data loading and cohort filtering
3. Preprocessing (per-image normalization)
4. Model architecture (multi-modal, multi-output 3D CNN / Transformer / Hybrid)
5. Cross-validated training with grouped, leakage-free splits
6. Evaluation: per-output accuracy and robustness across folds
7. Interpretation and conclusions

**Note on reproducibility:** this notebook requires access to restricted
imaging and phenotype data (see below) and a CUDA GPU, neither of which is
available in this environment. The cells are complete and ready to run
against that data; no results in this notebook are fabricated.

## 1. Problem and Dataset

**Task.** Given a 3D functional MRI contrast image (and optionally other
imaging modalities), predict several continuous cognitive/behavioral scores
per subject simultaneously.

**Dataset.** The data follows the variable-naming conventions of the ABCD
Study (Adolescent Brain Cognitive Development), a large longitudinal
neuroimaging study. Access to ABCD imaging and phenotype data requires a
Data Use Certification through the NIMH Data Archive, so the raw data is
**not included in this repository**. Instead, `config/config.yaml` and the
loading code below describe the expected structure:

- One `<modality>.npy` array per imaging modality: `[n_subjects, 1, D, H, W]`
  voxel intensities (here `D, H, W = 67, 67, 83`).
- One `<modality>_index.csv` per modality: a 0/1 flag per subject marking
  whether that subject's scan for that modality is usable (some subjects
  are missing some scan types).
- A phenotype CSV with the prediction targets, covariates, and identifiers
  used for grouped cross-validation (family ID, site ID, session ID).

To run this notebook end-to-end, place your own data under `./data/`
matching this layout and point `data_directory` in the config at it.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
from category_encoders import BinaryEncoder

from src.config import load_config
from src.data import (
    load_modalities,
    drop_samples_missing_any_modality,
    build_sample_filter,
    apply_sample_filter,
    to_tensor_datasets,
)
from src.preprocessing import PerImageNormalizer, PerImageSmoother
from src.models import ModelDesigner
from src.utils import count_parameters_by_layer
from src.training import train_multimodal_model

config_path = os.getenv('CONFIG_PATH', '../config/config.yaml')
config = load_config(config_path)
print(f"Loaded configuration from: {config_path}")
print(f"Modalities: {config.modality}")
print(f"Prediction targets: {[item['name'] for item in config.output]}")

## 2. Data Loading

Each modality contributes one voxel-intensity array and one validity index.
The phenotype data frame supplies the prediction targets, covariates, and
the identifiers (`rel_family_id`, site ID) used later for grouped
cross-validation.

In [ ]:
print("Loading imaging arrays and phenotype data...")

non_imaging_data = pd.read_csv(config.data_directory + config.data_frame)
image_arrays, modality_index = load_modalities(config.data_directory, config.modality)

output_columns = [item['name'] for item in config.output]
output_intervals = [item['interval'] for item in config.output]

print(f"Subjects in phenotype file: {len(non_imaging_data)}")
print(f"Loaded {len(config.modality)} modality array(s): "
      f"{[arr.shape for arr in image_arrays]}")

## 3. Cohort Filtering

A subject is kept only if **all** of the following hold:

- The prediction targets are non-missing.
- Every requested imaging modality has valid (non-missing) data for that
  subject.
- The imaging session falls within the requested time points.
- Covariates are non-missing (if covariates are enabled).
- The scanner manufacturer matches the requested filter (if enabled).
- Every prediction target falls inside its plausible-value interval (a
  basic sanity filter, e.g. a task-accuracy score must be between 0.6 and
  1.0), which guards against corrupted or placeholder values rather than
  behaviorally meaningful outliers.

All of these run **before** any train/val/test split, so the filtering
itself cannot leak test-set information into training -- it only removes
rows that are unusable for every subject regardless of which split they
would land in.

In [ ]:
if config.Drop_missing_modalities_flag:
    print("Checking for subjects with usable data in every modality...")
    complete_modality_mask = drop_samples_missing_any_modality(
        image_arrays, modality_index, config.modality
    )
else:
    complete_modality_mask = np.ones(len(non_imaging_data), dtype=bool)

covariate_columns = config.covariates if config.covariates_flag else None
time_points = config.time_point if config.time_point_flag else None
manufacturers = config.manufacture_filter if config.manufacture_flag else None
manufacturer_column = config.manufacture if config.manufacture_flag else None

sample_filter = build_sample_filter(
    non_imaging_data, output_columns, output_intervals, complete_modality_mask,
    covariate_columns=covariate_columns, time_points=time_points,
    manufacturers=manufacturers, manufacturer_column=manufacturer_column,
)

print(f"Subjects kept after all filters: {sample_filter.sum()} / {len(sample_filter)}")

covariates_raw = non_imaging_data[list(config.covariates)] if config.covariates_flag else None
image_arrays, modality_index, non_imaging_data, covariates_raw = apply_sample_filter(
    image_arrays, modality_index, non_imaging_data, sample_filter, covariates_raw
)

In [ ]:
# Identifiers used later for grouped cross-validation, and the final target frame
family_id = non_imaging_data['rel_family_id']
site_id = non_imaging_data['ab_g_dyn__design_site']
outputs = non_imaging_data[output_columns]

print(f"Final cohort size: {len(non_imaging_data)}")
print(f"Prediction targets: {output_columns}")
print(outputs.describe())

If covariates are enabled, categorical covariates (e.g. scanner
manufacturer) are binary-encoded here. This runs on the full filtered
cohort rather than per-fold because it only re-expresses categories as
bits -- it does not use any statistic (mean, count, etc.) that could differ
between train and test and leak information.

In [ ]:
if config.covariates_flag:
    covariates_data = covariates_raw.copy()
    for col in covariates_raw.columns:
        if covariates_raw[col].dtype == 'object' or str(covariates_raw[col].dtype) == 'category':
            encoded = BinaryEncoder(cols=[col]).fit_transform(covariates_raw[[col]])
            covariates_data = pd.concat([covariates_data.drop(columns=[col]), encoded], axis=1)
    print(f"Covariate matrix shape: {covariates_data.shape}")
else:
    covariates_data = None
    print("Covariates disabled for this run.")

print("Converting imaging arrays to PyTorch datasets...")
tensor_datasets = to_tensor_datasets(image_arrays)

## 4. Preprocessing: Per-Image Normalization

Each image is clipped to its own 1st-99th percentile and z-scored using
**its own** mean and standard deviation. Because every image is normalized
using only its own statistics (never a statistic computed across subjects),
this step is safe to apply before splitting into folds -- there is no
global training statistic that could leak into validation or test images.

In [ ]:
if config.normalizer_flag:
    normalizer = PerImageNormalizer(
        lower_quantile=config.image_normalizer_lower_quantile,
        upper_quantile=config.image_normalizer_upper_quantile,
        normalization_method=config.image_normalizer_normalization_method,
        modalities_to_clip=config.image_normalizer_modalities_to_clip,
        modalities_to_normalize=config.image_normalizer_modalities_to_normalize,
    )
    tensor_datasets = normalizer.apply_to_images(tensor_datasets, indices_to_include=modality_index)

if config.smoother_flag:
    smoother = PerImageSmoother(
        fwhm=config.image_smoother_fwhm,
        modalities_to_smooth=config.image_smoother_modalities_to_smooth,
    )
    tensor_datasets = smoother.apply_to_images(tensor_datasets, indices_to_include=modality_index)

## 5. Model Architecture

`ModelDesigner` builds one of three architectures from the same config
(`model_type` in `config.yaml`):

- **3DCNN** -- convolutional streams -> global average pooling -> FC head
- **3DVT** -- patch embedding -> transformer encoder -> FC head (3D Vision Transformer)
- **3DHB** -- convolutional streams -> patch embedding -> transformer encoder -> FC head (hybrid)

Every architecture ends in the same **multi-output head**: a shared
bottleneck layer feeds several small per-output heads, one per prediction
target. This is what lets a single model predict multiple cognitive scores
from one brain scan -- the shared bottleneck learns a general
brain-to-behavior representation, while each head specializes to its own
target.

In [ ]:
input_shape = tuple(config.input_shapes)
designer = ModelDesigner(
    architecture_type=config.model_type,
    layers_info=config.layers_info,
    n_input_streams=len(config.modality),
    input_shapes=[input_shape] * len(config.modality),
    fusion_type=config.fusion_type,
    covariate_flag=config.covariates_flag,
    covariate_dim=covariates_data.shape[1] if config.covariates_flag else 0,
    output_dim=outputs.shape[1],
    output_head_sizes=[64] * outputs.shape[1],
    bottleneck_dim=64,
)
model = designer.build_model()

print(f"Architecture: {config.model_type}  |  Fusion: {config.fusion_type}  |  Outputs: {outputs.shape[1]}")
count_parameters_by_layer(
    model,
    input_shapes=[input_shape] * len(config.modality),
    covariate_dim=covariates_data.shape[1] if config.covariates_flag else None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

## 6. Cross-Validated Training

**Split strategy.** Splits are grouped by `fold_strategy` in the config:

- `family` -- all scans from members of the same family stay together in
  the same split. This matters because siblings share genetics and
  environment, so letting them span train/test would let the model
  partly "recognize" family-level traits instead of generalizing to new
  people -- a classic source of inflated brain-imaging prediction accuracy.
- `site` -- all scans from the same collection site stay together, guarding
  against the model picking up site-specific scanner artifacts.

Within each fold, the train/validation split respects the same grouping,
and `MultiModal3DTrainer.prepare_data_splits` explicitly asserts that no
sample or group appears in more than one of train/val/test.

**Output scaling.** Targets are min-max scaled to `output_scale_range`
using statistics fit **only on that fold's training targets**, then
un-scaled before computing reported metrics -- so scaling cannot leak
val/test information into training either.

**Per-fold independence.** Model weights are re-initialized before every
fold, so each fold's reported performance reflects a model trained from
scratch on that fold's data, not a model that has already seen other
folds' training samples.

In [ ]:
grouping_variable = site_id if config.fold_strategy == 'site' else family_id
experiment_name = f"{config.model_type}_{'-'.join(config.modality)}_{'-'.join(output_columns)}"

print("Starting cross-validated training...")
results = train_multimodal_model(
    model=model,
    image_arrays=tensor_datasets,
    modality_index=modality_index,
    outputs=outputs,
    covariates=covariates_data,
    groups=grouping_variable,
    n_folds=config.n_folds,
    fold_strategy=config.fold_strategy,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    optimizer_type=config.optimizer_type,
    loss_function=config.loss_function,
    scheduler_type=config.scheduler_type,
    scheduler_params=config.scheduler_params,
    clip_grad=config.clip_grad,
    early_stopping_patience=config.early_stopping_patience,
    augmentation_config=config.augmentation_config,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir='../experiments',
    experiment_name=experiment_name,
    random_state=42,
    use_output_scaling=True,
    output_scale_range=(0, 1),
)

## 7. Training Results

`train_multimodal_model` saves, per fold, a loss/R² curve
(`training_history_fold_N.png`) and the best-validation-loss checkpoint.
Across folds, it saves:

- `predictions_per_output.png` -- true-vs-predicted scatter per target,
  points colored by fold
- `aggregate_results.png` -- validation loss and test R² per fold, plus the
  fold-averaged training curve
- `all_predictions.csv` -- every held-out test prediction, for custom analysis
- `results.json` -- per-output pooled metrics (R², MAE, RMSE)

These are written under `../experiments/<experiment_name>/`.

## 8. Evaluation: Accuracy and Robustness

Two complementary things matter here, not just one headline number:

**Accuracy** -- how well predictions track true values on held-out test
data, per output (R², MAE, RMSE). Brain-wide association studies are
notorious for modest effect sizes on individual differences, so R² should
be read in that context rather than compared to, say, image classification
benchmarks.

**Robustness (sensitivity to the split)** -- how much the reported R²
varies across the 10 folds. A model that gets R²=0.10 on every fold is a
more trustworthy result than one that averages to R²=0.10 by mixing folds
of 0.30 and -0.10 -- the second suggests the estimate is highly sensitive
to which subjects happened to land in the test set. `aggregate_results.png`
plots exactly this per-fold spread.

In [ ]:
summary = results['summary']
for name in results['output_names']:
    print(f"{name}: R^2 = {summary[f'{name}_r2']:.4f}, "
          f"MAE = {summary[f'{name}_mae']:.4f}, RMSE = {summary[f'{name}_rmse']:.4f}")

fold_r2 = results['test_metrics']
print(f"\nPer-fold test R^2: {[round(r, 4) for r in fold_r2]}")
print(f"Mean +/- SD across folds: {np.mean(fold_r2):.4f} +/- {np.std(fold_r2):.4f}")

## 9. Interpretation and Conclusions

- The model predicts multiple cognitive/behavioral outcomes from a single
  brain scan in one forward pass, sharing a bottleneck representation
  across outputs rather than training one model per target.
- Splits are grouped by family (or site) specifically to avoid the
  brain-wide-association-study pitfall of inflated accuracy from related
  subjects leaking across train/test.
- Reporting per-fold spread alongside the mean R² gives a more honest
  picture of how reliable the estimate is than a single train/test split
  would.

**Current scope and limitations:**
- The active configuration uses a single imaging modality
  (`nback_2bv0b_beta`); the multi-modal fusion path (`fusion_type`,
  additional entries under `modality`) is implemented and exercised by the
  architecture code, but multi-modality results depend on which modalities
  are enabled in `config.yaml`.
- `OutlierDetector`-style filtering of the imaging/behavioral data is
  configured (`outlier_flag` and related fields in `config.yaml`) but is
  not currently wired into the training loop -- see the README's
  "Known limitations" section.
- Reported R² values reflect squared Pearson correlation between
  predictions and true values on held-out test folds, not the coefficient
  of determination against the identity line; the two coincide only when
  predictions are unbiased.